Имеем 5 предприятий: $I = \{1, 2, 3, 4, 5\}$  
Имеем 3 склада: $J = \{1, 2, 3\}$  
Имеем 2 сортировочных станции: $K = \{1, 2\}$  
Имеем 4 заказа: $L = \{1, 2, 3, 4\}$

У предприятий есть мощность:  
$<A_1,...,A_5> \ (A_i) \ = \ <600, 500, 700, 400, 600>$  

У складов есть вместимость:  
$<W_1,...,W_3> \ (W_j) \ = \ <200, 500, 400>$  

У заказа есть (размер?):  
$<B_1,...,B_4> \ (B_l) \ = \ <100, 200, 200, 500>$  

У каждого товара есть стоимость производства (для всех предприятий она равна):  
$c_i^p: c^p \ = \ (1.3, 1.5, 1.2, 1.7, 1.1)$  

Еще у нас есть удельная стоимость перевозки товара с предприятия $i$ до склада $j$:  $c^1_{i,j}$    

Есть удельная стоимость перевозки товара со склада $j$ до станции $k$: $c^2_{j,k}$    

Есть удельная стоимость перевозки товра со станции $k$ до заказчика $l$: $c^3_{k,l}$  

Еще есть пропускная способность коммуникации $d^1_{i,j}$ (с предприятия до склада) 

И пропускная способность коммуникации $d^3_{k,l}$  

И удельная стоимость хранения на складе $s_1,...,s_4 \ (s_j) \ = \ <2, 1, 5, 3>$

#### Переменные  
$x_{i,j} \geq 0$ - объем продукции, перевозимый от предприятия $i$ до склада $j$  
$y_{j,k} \geq 0$ - объем продукции, перевозимый со склада $j$ до станции $k$  
$z_{k,l} \geq 0$ - объем продукции, перевозимый со станции $k$ до заказчика $l$  
$u_{i} \in \{0,1\}$ - 1, если предприятие задействовано и 0, если не задействовано 

#### Целевая функция  
Хотим минимизировать суммарные затраты на производство, транспортировку и хранение товара:  
$$\varphi = \sum_{i \in I} \sum_{j \in J} (c_i^p + c^1_{i, j}) \cdot x_{i, j} + \sum_{j \in J} \sum_{k \in K} (c^2_{j, k} + s_j) \cdot y_{j, k} + \sum_{k \in K} \sum_{l \in L} c^3_{k, l} \cdot z_{k, l} \rightarrow min$$

(Затраты на производство ($c_i^p \cdot x_{i, j}$) + доставка до склада ($c_{i, j}^1 \cdot x_{i, j} $))  
(Затраты на доставку от склада до станции ($c^2_{j,k} \cdot y_{j,k}$) + хранение на складе ($s_j \cdot y_{j, k}$))  
(Затраты на доставку от станции до потребителя)  

#### Ограничения

Равенство объема произведенной и доставленной продукции  
$$\sum_{i \in I} x_{i, j} = \sum_{k \in K} y_{j, k} \ \forall j \in J$$

Все товары со складов проходят через сортировочные центры  
$$\sum_{j \in J} y_{j, k} = \sum_{l \in L} z_{k, l} \ \forall k \in K$$

Все заказчики в полной мере удовлетворены  
$$\sum_{k \in K}z_{k, l} = B_{l} \ \forall l \in L$$

**Даже работающее** предприятие не может выпустить больше, чем может  
$$\sum_{j \in J}x_{i, j} \leq A_i \cdot u_i \ \forall i \in I$$

Склады имеют ограниченный объем  
$$\sum_{k \in K}y_{j, k} \leq W_j \ \forall j \in J$$

Пропускные способности коммуникаций 
$x_{i, j} \leq d^1_{i, j} \ \forall i \in I, \ \forall j \in J$
$z_{k, l} \leq d^3_{k, l} \ \forall k \in K, \ \forall l \in L$

Условия неотрицательности
$x_{i, j} \geq 0, y_{j, k} \geq 0, z_{k, l} \geq 0, u_i \in \{0, 1\}$

In [1]:
from ortools.linear_solver import pywraplp

num_E = 5 
num_W = 3  
num_S = 2  
num_C = 4  

A = [600, 500, 700, 400, 600]
C_prod = [1.3, 1.5, 1.2, 1.7, 1.1]
W_cap = [200, 500, 400]
B = [100, 200, 200, 500]

D1 = [
    [150, 500, None],
    [100, None, 400],
    [50, 50, None],
    [None, 100, 100],
    [100, None, 50]
]

C1 = [
    [1, 2, None],
    [2, None, 1],
    [3, 3, None],
    [None, 3, 2],
    [2, None, 3]
]

C2 = [
    [2, 3],
    [1, 4],
    [3, 1]
]

C3 = [
    [2, 1, 3, 4],
    [3, 2, 3, 1]
]

D3 = [
    [50, 50, 10, 300],
    [70, 150, 200, 250]
]

# задача 3
F_fixed_million = [1, 2, 3, 2.5, 1.5]

# задача 4
S_storage = [2, 1.5, 3]

def solve_lab_task(task_number):

    solver = pywraplp.Solver.CreateSolver('SCIP')
    if not solver:
        print("Ошибка.")
        return

    x = {}
    for i in range(num_E):
        for j in range(num_W):
            ub = D1[i][j] if D1[i][j] is not None else 0
            x[i, j] = solver.NumVar(0, ub, f'x_{i}_{j}')

    y = {}
    for j in range(num_W):
        for k in range(num_S):
            y[j, k] = solver.NumVar(0, solver.infinity(), f'y_{j}_{k}')

    z = {}
    for k in range(num_S):
        for l in range(num_C):
            z[k, l] = solver.NumVar(0, D3[k][l], f'z_{k}_{l}')

    u = {}
    for i in range(num_E):
        u[i] = solver.IntVar(0, 1, f'u_{i}')


    for j in range(num_W):
        inflow = sum(x[i, j] for i in range(num_E))
        outflow = sum(y[j, k] for k in range(num_S))
        
        if task_number == 2 and j == 0:
            solver.Add(inflow + 100 == outflow)
        else:
            solver.Add(inflow == outflow)

    for k in range(num_S):
        inflow = sum(y[j, k] for j in range(num_W))
        outflow = sum(z[k, l] for l in range(num_C))
        solver.Add(inflow == outflow)

    for l in range(num_C):
        solver.Add(sum(z[k, l] for k in range(num_S)) == B[l])

    for i in range(num_E):
        solver.Add(sum(x[i, j] for j in range(num_W)) <= A[i] * u[i])

    for j in range(num_W):
        solver.Add(sum(y[j, k] for k in range(num_S)) <= W_cap[j])

    # В задачах 3 и 4 это ограничение снимается
    if task_number in [1, 2]:
        solver.Add(sum(u[i] for i in range(num_E)) <= 2)


    obj = solver.Objective()
    
    for i in range(num_E):
        for j in range(num_W):
            if D1[i][j] is not None:
                obj.SetCoefficient(x[i, j], C_prod[i] + C1[i][j])
            else:
                obj.SetCoefficient(x[i, j], 0)

    for j in range(num_W):
        for k in range(num_S):
            cost = C2[j][k]
            if task_number == 4:
                cost += S_storage[j] 
            obj.SetCoefficient(y[j, k], cost)

    for k in range(num_S):
        for l in range(num_C):
            obj.SetCoefficient(z[k, l], C3[k][l])

    if task_number == 3:
        for i in range(num_E):
            obj.SetCoefficient(u[i], F_fixed_million[i] * 1000)

    obj.SetMinimization()


    status = solver.Solve()

    if status == pywraplp.Solver.OPTIMAL:
        print(f"ИТОГОВАЯ СУММА: {obj.Value():.2f} тыс. руб.\n")
        
        prod_cost = sum(C_prod[i] * x[i, j].solution_value() for i in range(num_E) for j in range(num_W) if D1[i][j] is not None)
        trans_c1 = sum(C1[i][j] * x[i, j].solution_value() for i in range(num_E) for j in range(num_W) if D1[i][j] is not None)
        trans_c2 = sum(C2[j][k] * y[j, k].solution_value() for j in range(num_W) for k in range(num_S))
        trans_c3 = sum(C3[k][l] * z[k, l].solution_value() for k in range(num_S) for l in range(num_C))
        
        storage_cost = 0
        if task_number == 4:
            storage_cost = sum(S_storage[j] * y[j, k].solution_value() for j in range(num_W) for k in range(num_S))

        fixed_cost = 0
        if task_number == 3:
            fixed_cost = sum(F_fixed_million[i] * 1000 * u[i].solution_value() for i in range(num_E))

        print(f"1. Производство:                 {prod_cost:10.2f} тыс. руб.")
        print(f"2. Перевозка Предприятия->Склады:    {trans_c1:10.2f} тыс. руб.")
        print(f"3. Перевозка Склады->Станции:    {trans_c2:10.2f} тыс. руб.")
        print(f"4. Перевозка Станции->Заказчики:   {trans_c3:10.2f} тыс. руб.")
        if task_number == 3:
            print(f"5. Организация производства:     {fixed_cost:10.2f} тыс. руб.")
        if task_number == 4:
            print(f"5. Хранение на складах:          {storage_cost:10.2f} тыс. руб.")
        print("-" * 40)
        print(f"ИТОГО:                           {prod_cost + trans_c1 + trans_c2 + trans_c3 + storage_cost + fixed_cost:10.2f} тыс. руб.")

        active_plants = [i + 1 for i in range(num_E) if u[i].solution_value() > 0.5]
        print(f"\nИспользуемые предприятия: {active_plants}")
        
        if task_number == 4:
            print("\nОбъемы грузопотоков через склады:")
            for j in range(num_W):
                vol_j = sum(y[j, k].solution_value() for k in range(num_S))
                print(f"  Склад {j+1}: {vol_j:.1f} тыс. тонн (тариф хранения: {S_storage[j]} руб/т)")

    else:
        print("Решение не найдено или задача неразрешима.")
    print("\n")



if __name__ == '__main__':
    solve_lab_task(1)
    solve_lab_task(2)
    solve_lab_task(3)
    solve_lab_task(4)

ИТОГОВАЯ СУММА: 6810.00 тыс. руб.

1. Производство:                    1390.00 тыс. руб.
2. Перевозка Предприятия->Склады:       1450.00 тыс. руб.
3. Перевозка Склады->Станции:       1400.00 тыс. руб.
4. Перевозка Станции->Заказчики:      2570.00 тыс. руб.
----------------------------------------
ИТОГО:                              6810.00 тыс. руб.

Используемые предприятия: [1, 2]


ИТОГОВАЯ СУММА: 6520.00 тыс. руб.

1. Производство:                    1250.00 тыс. руб.
2. Перевозка Предприятия->Склады:       1300.00 тыс. руб.
3. Перевозка Склады->Станции:       1400.00 тыс. руб.
4. Перевозка Станции->Заказчики:      2570.00 тыс. руб.
----------------------------------------
ИТОГО:                              6520.00 тыс. руб.

Используемые предприятия: [1, 2]


ИТОГОВАЯ СУММА: 9810.00 тыс. руб.

1. Производство:                    1390.00 тыс. руб.
2. Перевозка Предприятия->Склады:       1450.00 тыс. руб.
3. Перевозка Склады->Станции:       1400.00 тыс. руб.
4. Перевозка Станции->З